In [1]:
import numpy as np
import os
import h5py
import sys

from tqdm import tqdm

import torch

sys.path.insert(0, '../src/')
import vids_segm_cld.vids_segm_pl as vids

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# path_to_data_test     = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/TEST/'
# path_to_data_test = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/VAL/'
path_to_data_test = '/home/paul/Desktop/data_ped_unc/echonet_dynamic_preprocessed/VAL/'
# path_to_data_train    = '/home/paul/Desktop/data_ped_unc/echonet_ped_preprocessed/TRAIN/'
path_to_data_train    = '/home/paul/Desktop/data_ped_unc/echonet_dynamic_preprocessed/TRAIN/'
path_to_inference_out = '/home/paul/Desktop/data_ped_unc/predictions_dyn/vids/'

# Number of stochastic theta samples per image
n_samples = 20
# Number of training images used as context for the inference network
n_context = 32

os.makedirs(path_to_inference_out, exist_ok=True)

In [3]:
@torch.no_grad()
def sample_softmax_vids(model, x, train_summary, n_samples=20):
    """
    Generates softmax samples from VIDS for a single test image,
    conditioned on a precomputed training summary.
    Returns shape: (n_samples, n_classes, H, W)
    """
    model.eval()

    test_emb     = model.compute_embeddings(x)
    test_summary = model.aggregate_single_image_embedding(test_emb)

    mu, log_sigma = model.inference_net(train_summary, test_summary)
    sigma = torch.exp(log_sigma)

    eps           = torch.randn(n_samples, mu.size(0), device=mu.device)
    theta_samples = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps

    logits = model.prediction_head(test_emb, theta_samples).squeeze()  # (n_samples, n_classes, H, W)
    return logits

In [4]:
def load_vids_model(ckpt_path):
    embedding_dim    = 64
    num_classes      = 2
    n_channels       = 1
    inference_hidden = [512, 512, 256, 256]

    embedding_net = vids.UNetDenseEmbedding(
        n_channels=n_channels,
        embedding_dim=embedding_dim,
        bilinear=False,
    )

    model = vids.VIDS(
        embedding_net=embedding_net,
        embedding_dim=embedding_dim,
        output_dim=num_classes,
        task="segmentation",
        num_classes=num_classes,
        inference_hidden_dims=inference_hidden,
        env_train_size=32,
        env_test_size=10,
    )

    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt.get("state_dict", ckpt)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

In [5]:
# Fill in checkpoint path before running
model_path = '/home/paul/Desktop/data_ped_unc/models_eval/vids/adults_norm/version_0/checkpoints/epoch=21-step=5368.ckpt'
# model_path = '/home/paul/Desktop/data_ped_unc/models_eval/vids/adults_norm/version_1/checkpoints/epoch=16-step=4148.ckpt'
# model_path = '/home/paul/Desktop/data_ped_unc/models_eval/vids/adults_norm/version_2/checkpoints/epoch=12-step=3172.ckpt'
# model_path = '/home/paul/Desktop/data_ped_unc/models_eval/vids/adults_norm/version_3/checkpoints/epoch=15-step=3904.ckpt'

print(f"Loading VIDS model from {model_path} ...")
model = load_vids_model(model_path)
print(f"Model loaded. Will draw {n_samples} samples per image.")

Loading VIDS model from /home/paul/Desktop/data_ped_unc/models_eval/vids/adults_norm/version_0/checkpoints/epoch=21-step=5368.ckpt ...
Model loaded. Will draw 20 samples per image.


In [6]:
def load_training_context(train_dir, n_images):
    context_images = []
    for f in sorted(os.listdir(train_dir)):
        if len(context_images) >= n_images:
            break
        try:
            with h5py.File(os.path.join(train_dir, f), 'r') as data:
                for phase in ['ed', 'es']:
                    if phase in data and len(context_images) < n_images:
                        context_images.append(data[phase]['image'][()])
        except Exception:
            continue

    context_tensor = torch.tensor(
        np.stack(context_images), dtype=torch.float32
    ).unsqueeze(1).to(device)  # (N, 1, H, W)
    return context_tensor


print(f"Loading {n_context} training context images...")
train_context = load_training_context(path_to_data_train, n_context)
print(f"  Context tensor shape: {train_context.shape}")

print("Precomputing training summary embedding (one-time cost)...")
with torch.no_grad():
    train_emb     = model.compute_embeddings(train_context)
    train_summary = model.aggregate_embeddings(train_emb)  # (D,)
print(f"  Training summary shape: {train_summary.shape}")

Loading 32 training context images...
  Context tensor shape: torch.Size([32, 1, 112, 112])
Precomputing training summary embedding (one-time cost)...
  Training summary shape: torch.Size([128])


In [7]:
file_names = sorted(os.listdir(path_to_data_test))
print(f"Running inference on {len(file_names)} files...")

for file in tqdm(file_names):
    full_filepath = os.path.join(path_to_data_test, file)
    out_filepath  = os.path.join(path_to_inference_out, file)

    with h5py.File(full_filepath, 'r') as src, h5py.File(out_filepath, 'w') as dst:
        phases_present = [p for p in ['ed', 'es'] if p in src]

        for phase in phases_present:
            img = src[phase]['image'][()]  # (H, W)
            img_tensor = torch.Tensor(img).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, H, W)

            # VIDS stochastic inference: returns softmax samples (n_samples, n_classes, H, W)
            softmax_samples = sample_softmax_vids(model, img_tensor, train_summary, n_samples=n_samples)

            logits_arr = softmax_samples.cpu().numpy()  # (n_samples, n_classes, H, W)

            grp = dst.require_group(phase)
            grp.create_dataset('logits', data=logits_arr)

print("Done. Outputs saved to", path_to_inference_out)

Running inference on 1186 files...


100%|██████████| 1186/1186 [00:10<00:00, 114.92it/s]

Done. Outputs saved to /home/paul/Desktop/data_ped_unc/predictions_dyn/vids/
